# Évaluation Clinique & Analyse d’Erreurs

## Objectif
Ce notebook centralise l’analyse finale des performances du pipeline, avec un focus particulier sur :
- les classes critiques,
- les erreurs cliniquement sensibles,
- la justification du modèle champion,
- les limites méthodologiques et cliniques.

## Portée
Ce notebook ne réalise **aucun nouvel entraînement**.  
Il exploite uniquement les artefacts exportés par les notebooks de benchmark.

## Rappel clinique
Le projet reste un outil de **triage non diagnostique**.  
Il ne remplace ni un avis médical, ni une évaluation psychiatrique, ni une décision clinique.

In [7]:
import sys
from pathlib import Path

current = Path.cwd().resolve()

if current.name == "notebooks":
    PROJECT_ROOT = current.parent
elif (current / "src").exists():
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = current

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config.paths import *

In [ ]:
# ============================================================
# PHASE 0 — PATH CONFIGURATION
# ============================================================

from pathlib import Path
import os

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        PROJECT_ROOT 
    )
)

DATA_DIR = PROJECT_ROOT / "data"
DATA_CLEAN_DIR = DATA_DIR / "clean"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

# Keep old variable names to avoid breaking notebook code
CLASSICAL_REPORTS_DIR = TABLES_DIR / "classical"
COMPARISON_REPORTS_DIR = TABLES_DIR / "classical"

CLINICAL_FIGURES_DIR = FIGURES_DIR / "clinical"
CLINICAL_TABLES_DIR = TABLES_DIR / "clinical"

# --- New transformer paths ---
TRANSFORMERS_TABLES_DIR = TABLES_DIR / "transformers"
BERT_TABLES_DIR = TRANSFORMERS_TABLES_DIR / "bert"
MENTAL_BERT_TABLES_DIR = TRANSFORMERS_TABLES_DIR / "mental_bert"
TRANSFORMER_COMPARISON_DIR = TRANSFORMERS_TABLES_DIR / "comparison"

CLEAN_DATA_PATH = DATA_CLEAN_DIR / "mental_health_detection_clean.csv"

FINAL_TEST_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "final_test_metrics.csv"
CLASS_RECALL_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "class_recall_results.csv"
FINAL_PREDICTIONS_PATH = CLASSICAL_REPORTS_DIR / "final_test_predictions.csv"

NORMAL_CV_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "normal_cv_summary.csv"
NESTED_CV_RESULTS_PATH = CLASSICAL_REPORTS_DIR / "nested_cv_summary.csv"
COMPARISON_RESULTS_PATH = COMPARISON_REPORTS_DIR / "classical_model_comparison.csv"

# --- Transformer artifact paths ---
BERT_METRICS_PATH = BERT_TABLES_DIR / "bert_base_raw_metrics.csv"
BERT_CLASS_RECALL_PATH = BERT_TABLES_DIR / "bert_base_raw_class_recall.csv"
BERT_PREDICTIONS_PATH = BERT_TABLES_DIR / "bert_base_raw_predictions.csv"

MENTAL_BERT_METRICS_PATH = MENTAL_BERT_TABLES_DIR / "mental_bert_raw_metrics.csv"
MENTAL_BERT_CLASS_RECALL_PATH = MENTAL_BERT_TABLES_DIR / "mental_bert_raw_class_recall.csv"
MENTAL_BERT_PREDICTIONS_PATH = MENTAL_BERT_TABLES_DIR / "mental_bert_raw_predictions.csv"

TRANSFORMER_MODEL_COMPARISON_PATH = (
    TRANSFORMER_COMPARISON_DIR / "transformer_model_comparison.csv"
)
TRANSFORMER_TEST_RESULTS_PATH = (
    TRANSFORMER_COMPARISON_DIR / "transformer_test_results.csv"
)

for directory in [
    CLINICAL_FIGURES_DIR,
    CLINICAL_TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("✅ Clinical notebook paths ready")
print(f"PROJECT_ROOT                      : {PROJECT_ROOT}")
print(f"FINAL_TEST_RESULTS_PATH           : {FINAL_TEST_RESULTS_PATH}")
print(f"CLASS_RECALL_RESULTS_PATH         : {CLASS_RECALL_RESULTS_PATH}")
print(f"FINAL_PREDICTIONS_PATH            : {FINAL_PREDICTIONS_PATH}")
print(f"NORMAL_CV_RESULTS_PATH            : {NORMAL_CV_RESULTS_PATH} | exists={NORMAL_CV_RESULTS_PATH.exists()}")
print(f"NESTED_CV_RESULTS_PATH            : {NESTED_CV_RESULTS_PATH} | exists={NESTED_CV_RESULTS_PATH.exists()}")
print(f"COMPARISON_RESULTS_PATH           : {COMPARISON_RESULTS_PATH} | exists={COMPARISON_RESULTS_PATH.exists()}")
print(f"TRANSFORMER_MODEL_COMPARISON_PATH : {TRANSFORMER_MODEL_COMPARISON_PATH} | exists={TRANSFORMER_MODEL_COMPARISON_PATH.exists()}")
print(f"TRANSFORMER_TEST_RESULTS_PATH     : {TRANSFORMER_TEST_RESULTS_PATH} | exists={TRANSFORMER_TEST_RESULTS_PATH.exists()}")
print(f"CLEAN_DATA_PATH                   : {CLEAN_DATA_PATH} | exists={CLEAN_DATA_PATH.exists()}")

✅ Clinical notebook paths ready
PROJECT_ROOT                      : C:\Users\anafi\Desktop\final_project_jedha
FINAL_TEST_RESULTS_PATH           : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\final_test_metrics.csv
CLASS_RECALL_RESULTS_PATH         : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\class_recall_results.csv
FINAL_PREDICTIONS_PATH            : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\final_test_predictions.csv
NORMAL_CV_RESULTS_PATH            : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\normal_cv_summary.csv | exists=True
NESTED_CV_RESULTS_PATH            : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\nested_cv_summary.csv | exists=True
COMPARISON_RESULTS_PATH           : C:\Users\anafi\Desktop\final_project_jedha\reports\tables\classical\classical_model_comparison.csv | exists=True
TRANSFORMER_MODEL_COMPARISON_PATH : C:\Users\anafi\Desktop\final_project_

## Chargement des artefacts exportés

Cette section charge les fichiers produits par le benchmark classique afin de centraliser l’analyse finale dans un seul notebook.

In [9]:
# ============================================================
# PHASE 0 — IMPORTS & GLOBAL SETTINGS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from IPython.display import display
from sklearn.metrics import confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TEXT_COL = "body"
MASKED_COL = "body_masked"
TARGET_COL = "category"

PLOTLY_TEMPLATE = "plotly_white"
CRITICAL_LABELS = ["Bipolar", "Schizophrenia"]

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_rows", 100)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

print("✅ Imports loaded")
print(f"TEXT_COL   = {TEXT_COL}")
print(f"MASKED_COL = {MASKED_COL}")
print(f"TARGET_COL = {TARGET_COL}")
print(f"CRITICAL_LABELS = {CRITICAL_LABELS}")

✅ Imports loaded
TEXT_COL   = body
MASKED_COL = body_masked
TARGET_COL = category
CRITICAL_LABELS = ['Bipolar', 'Schizophrenia']


In [10]:
def highlight_critical_label(value):
    if value in CRITICAL_LABELS:
        return "background-color: #fff3cd; font-weight: bold;"
    return ""

## Chargement des artefacts exportés

Cette section charge les fichiers produits par le benchmark classique afin de centraliser l’analyse finale dans un seul notebook.

In [11]:
# ============================================================
# PHASE 1 — LOAD EXPORTED ARTIFACTS
# ============================================================

def load_first_existing(paths):
    for path in paths:
        if path.exists():
            df = pd.read_csv(path)
            print(f"✅ Loaded: {path.name} | shape={df.shape}")
            return df, path
    print("⚠️ Missing file candidates:")
    for path in paths:
        print(f"   - {path}")
    return None, None

final_test_metrics, final_test_metrics_path = load_first_existing([
    FINAL_TEST_RESULTS_PATH,
])

class_recall_df, class_recall_df_path = load_first_existing([
    CLASS_RECALL_RESULTS_PATH,
])

final_predictions_df, final_predictions_df_path = load_first_existing([
    FINAL_PREDICTIONS_PATH,
])

nested_cv_summary, nested_cv_summary_path = load_first_existing([
    NESTED_CV_RESULTS_PATH,
])

light_cv_summary, light_cv_summary_path = load_first_existing([
    NORMAL_CV_RESULTS_PATH,
])

comparison_df, comparison_df_path = load_first_existing([
    COMPARISON_RESULTS_PATH,
    CLINICAL_TABLES_DIR / "global_comparison_for_clinical_review.csv",
])

df_clean, df_clean_path = load_first_existing([
    CLEAN_DATA_PATH,
])

# --- Optional transformer artifacts ---
bert_metrics_df, bert_metrics_path = load_first_existing([
    BERT_METRICS_PATH,
])

bert_class_recall_df, bert_class_recall_path = load_first_existing([
    BERT_CLASS_RECALL_PATH,
])

bert_predictions_df, bert_predictions_path = load_first_existing([
    BERT_PREDICTIONS_PATH,
])

mental_bert_metrics_df, mental_bert_metrics_path = load_first_existing([
    MENTAL_BERT_METRICS_PATH,
])

mental_bert_class_recall_df, mental_bert_class_recall_path = load_first_existing([
    MENTAL_BERT_CLASS_RECALL_PATH,
])

mental_bert_predictions_df, mental_bert_predictions_path = load_first_existing([
    MENTAL_BERT_PREDICTIONS_PATH,
])

transformer_comparison_df, transformer_comparison_path = load_first_existing([
    TRANSFORMER_MODEL_COMPARISON_PATH,
    TRANSFORMER_TEST_RESULTS_PATH,
])

required_artifacts = {
    "final_test_metrics": final_test_metrics,
    "final_predictions_df": final_predictions_df,
    "nested_cv_summary": nested_cv_summary,
    "light_cv_summary": light_cv_summary,
    "df_clean": df_clean,
}

missing_required = [name for name, obj in required_artifacts.items() if obj is None]
if missing_required:
    raise ValueError(f"Missing required artifacts: {missing_required}")

print("\n✅ Artifact loading step completed")

✅ Loaded: final_test_metrics.csv | shape=(1, 5)
✅ Loaded: class_recall_results.csv | shape=(7, 5)
✅ Loaded: final_test_predictions.csv | shape=(2255, 5)
✅ Loaded: nested_cv_summary.csv | shape=(10, 11)
✅ Loaded: normal_cv_summary.csv | shape=(10, 11)
✅ Loaded: classical_model_comparison.csv | shape=(20, 12)
✅ Loaded: mental_health_detection_clean.csv | shape=(11271, 3)
✅ Loaded: bert_base_raw_metrics.csv | shape=(1, 6)
✅ Loaded: bert_base_raw_class_recall.csv | shape=(7, 5)
✅ Loaded: bert_base_raw_predictions.csv | shape=(1691, 4)
✅ Loaded: mental_bert_raw_metrics.csv | shape=(1, 6)
✅ Loaded: mental_bert_raw_class_recall.csv | shape=(7, 5)
✅ Loaded: mental_bert_raw_predictions.csv | shape=(1691, 4)
✅ Loaded: transformer_test_results.csv | shape=(2, 7)

✅ Artifact loading step completed


In [12]:
display(final_test_metrics)
display(class_recall_df.head())
display(final_predictions_df.head())

,champion_model,text_variant,f1_macro,recall_macro,critical_recall
0,LinearSVC_balanced,raw,0.778927,0.778686,0.698068


,label,precision,recall,f1-score,support
0,ADHD,0.877108,0.907731,0.892157,401.0
1,Autism,0.843333,0.846154,0.844741,299.0
2,BPD,0.817337,0.814815,0.816074,324.0
3,Anxiety,0.808108,0.778646,0.793103,384.0
4,Bipolar,0.810811,0.739726,0.773639,365.0


,text,text_used_for_model,y_true,y_pred,is_correct
0,My first day of JR year starts tomorrow and I cant do it Well I can but its very difficult for me mentally I was online from 7th9th grade went in person for 10th grade in a different area and have no friends I dont even have any from middle school or elementary either since I lived far away from...,My first day of JR year starts tomorrow and I cant do it Well I can but its very difficult for me mentally I was online from 7th9th grade went in person for 10th grade in a different area and have no friends I dont even have any from middle school or elementary either since I lived far away from...,Anxiety,Anxiety,True
1,Its one of those days where I feel like garbage and completely hate myself for it\n\n\n\nI dont even know where to begin\n\n\n\nI came home from work after trying to convince myself that everyone likes me or at the very least doesnt hate me But all I get instead is just a non stop paranoid strea...,Its one of those days where I feel like garbage and completely hate myself for it\n\n\n\nI dont even know where to begin\n\n\n\nI came home from work after trying to convince myself that everyone likes me or at the very least doesnt hate me But all I get instead is just a non stop paranoid strea...,Anxiety,Anxiety,True
2,a while back i stumbled across a ted talk about adhd and the speaker open his lecture by listing all the hobbies he has he had a lot but it made me realize that i have a lot too\n\nmy hobbies\n\nmusic love to listen to music also play guitar and use to play drums\n\nvideo games play dota 2 mostl...,a while back i stumbled across a ted talk about adhd and the speaker open his lecture by listing all the hobbies he has he had a lot but it made me realize that i have a lot too\n\nmy hobbies\n\nmusic love to listen to music also play guitar and use to play drums\n\nvideo games play dota 2 mostl...,ADHD,ADHD,True
3,I was having cardiac anxiety amp after 1 year of struggle more than 20 Lipid profile tests consultation with 15 cardiologists i finally gave up on the thoughts of getting heart problems amp it successfully resolved my cardiophobia hypochondria\n\nNow after 2 months something happened at my famil...,I was having cardiac anxiety amp after 1 year of struggle more than 20 Lipid profile tests consultation with 15 cardiologists i finally gave up on the thoughts of getting heart problems amp it successfully resolved my cardiophobia hypochondria\n\nNow after 2 months something happened at my famil...,Anxiety,Anxiety,True
4,My anxiety has tormented me for the most of my living time on this earth for as long as my memory allows as soon as preteen But recently its been at its worst I feel like a failure in everything that I do When my anxiety triggers its like I lose all common sense and how to act Like even driving ...,My anxiety has tormented me for the most of my living time on this earth for as long as my memory allows as soon as preteen But recently its been at its worst I feel like a failure in everything that I do When my anxiety triggers its like I lose all common sense and how to act Like even driving ...,Anxiety,Anxiety,True


In [13]:
# ============================================================
# PHASE 2 — REBUILD CHAMPION CONTEXT
# ============================================================

if nested_cv_summary.empty:
    raise ValueError("nested_cv_summary is empty.")

champion_candidates = nested_cv_summary.copy()

champion_candidates = champion_candidates.sort_values(
    by=["robust_score", "critical_recall_mean", "f1_macro_mean"],
    ascending=False,
).reset_index(drop=True)

CHAMPION_ROW = champion_candidates.iloc[0].to_dict()
CHAMPION_MODEL_NAME = CHAMPION_ROW["model"]
CHAMPION_TEXT_VARIANT = CHAMPION_ROW["text_variant"]

print("✅ Champion context rebuilt")
print("CHAMPION_MODEL_NAME :", CHAMPION_MODEL_NAME)
print("CHAMPION_TEXT_VARIANT:", CHAMPION_TEXT_VARIANT)

display(champion_candidates.head())

✅ Champion context rebuilt
CHAMPION_MODEL_NAME : LinearSVC_balanced
CHAMPION_TEXT_VARIANT: raw


,model,f1_macro_mean,recall_macro_mean,critical_recall_mean,f1_macro_std,recall_macro_std,critical_recall_std,robust_score,robust_rank,protocol,text_variant
0,LinearSVC_balanced,0.768516,0.767001,0.684379,0.006075,0.006391,0.018273,0.734407,1,nested_cv,raw
1,LogReg_balanced,0.724886,0.729826,0.708881,0.003861,0.003367,0.012813,0.719966,2,nested_cv,raw
2,LinearSVC_plain,0.757862,0.753905,0.659020,0.008144,0.007790,0.026159,0.717138,3,nested_cv,raw
3,LogReg_plain,0.753981,0.748879,0.630051,0.013695,0.013482,0.034486,0.702878,4,nested_cv,raw
4,LinearSVC_balanced,0.692175,0.691903,0.597883,0.020567,0.020627,0.027198,0.654376,1,nested_cv,masked


## Comparaison globale des résultats

Avant d’entrer dans l’analyse d’erreurs, nous résumons les performances globales afin de replacer le modèle champion dans le contexte complet du benchmark.

In [14]:
# ============================================================
# PHASE 3 — GLOBAL COMPARISON TABLE
# ============================================================

def standardize_transformer_comparison(df_input):
    """
    Standardize transformer comparison columns to match the clinical notebook.
    """
    if df_input is None or df_input.empty:
        return None

    df = df_input.copy()

    rename_map = {
        "model_name": "model",
        "critical_recall": "critical_recall_mean",
        "f1_macro": "f1_macro_mean",
        "recall_macro": "recall_macro_mean",
    }
    df = df.rename(columns=rename_map)

    if "model" not in df.columns:
        return None

    if "text_variant" not in df.columns:
        df["text_variant"] = "raw"

    if "protocol" not in df.columns:
        df["protocol"] = "transformer_test"

    if "robust_score" not in df.columns:
        if {
            "f1_macro_mean",
            "recall_macro_mean",
            "critical_recall_mean",
        }.issubset(df.columns):
            df["robust_score"] = (
                0.4 * df["f1_macro_mean"]
                + 0.2 * df["recall_macro_mean"]
                + 0.4 * df["critical_recall_mean"]
            )
        else:
            df["robust_score"] = np.nan

    for col in [
        "f1_macro_mean",
        "recall_macro_mean",
        "critical_recall_mean",
        "robust_score",
    ]:
        if col not in df.columns:
            df[col] = np.nan

    df["robust_rank"] = (
        df["robust_score"].rank(method="dense", ascending=False).astype("Int64")
    )

    keep_cols = [
        "model",
        "text_variant",
        "protocol",
        "f1_macro_mean",
        "recall_macro_mean",
        "critical_recall_mean",
        "robust_score",
        "robust_rank",
    ]

    return df[keep_cols].copy()

# --- Classical comparison ---
if comparison_df is None or comparison_df.empty:
    comparison_df = pd.concat(
        [light_cv_summary, nested_cv_summary],
        axis=0,
        ignore_index=True,
    )

# --- Transformer comparison ---
transformer_comparison_std = standardize_transformer_comparison(transformer_comparison_df)

if transformer_comparison_std is not None and not transformer_comparison_std.empty:
    classical_cols = [
        "model",
        "text_variant",
        "protocol",
        "f1_macro_mean",
        "recall_macro_mean",
        "critical_recall_mean",
        "robust_score",
        "robust_rank",
    ]

    classical_comparison_std = comparison_df.copy()

    for col in classical_cols:
        if col not in classical_comparison_std.columns:
            classical_comparison_std[col] = np.nan

    classical_comparison_std = classical_comparison_std[classical_cols]

    comparison_df = pd.concat(
        [classical_comparison_std, transformer_comparison_std],
        axis=0,
        ignore_index=True,
    )

comparison_df = comparison_df.sort_values(
    by=["protocol", "text_variant", "robust_score"],
    ascending=[True, True, False],
).reset_index(drop=True)

display(comparison_df.head(20))
print("Shape:", comparison_df.shape)

comparison_df.to_csv(
    CLINICAL_TABLES_DIR / "global_comparison_for_clinical_review.csv",
    index=False,
)

print("✅ Global comparison table saved")

,model,text_variant,protocol,f1_macro_mean,recall_macro_mean,critical_recall_mean,robust_score,robust_rank
0,LinearSVC_balanced,masked,light_cv,0.692175,0.691903,0.597883,0.663806,1
1,LogReg_balanced,masked,light_cv,0.683549,0.683117,0.593510,0.656408,2
2,LinearSVC_plain,masked,light_cv,0.685336,0.681800,0.564224,0.647942,3
3,LogReg_plain,masked,light_cv,0.674757,0.670416,0.530150,0.630072,4
4,MultinomialNB,masked,light_cv,0.492918,0.516509,0.311855,0.445677,5
5,LinearSVC_balanced,raw,light_cv,0.768516,0.767001,0.684379,0.742820,1
6,LogReg_balanced,raw,light_cv,0.755790,0.755371,0.678057,0.732344,2
7,LinearSVC_plain,raw,light_cv,0.766738,0.761514,0.651268,0.730530,3
8,LogReg_plain,raw,light_cv,0.753981,0.748879,0.630051,0.715271,4
9,MultinomialNB,raw,light_cv,0.545324,0.566147,0.350395,0.493092,5


Shape: (22, 8)
✅ Global comparison table saved


In [15]:
fig = px.bar(
    comparison_df,
    x="model",
    y="robust_score",
    color="text_variant",
    facet_col="protocol",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    title="Vue consolidée des performances des modèles",
    hover_data=[
        "f1_macro_mean",
        "recall_macro_mean",
        "critical_recall_mean",
        "robust_rank",
    ],
)

fig.update_layout(
    xaxis_title="Modèle",
    yaxis_title="Robust score",
    title_x=0.5,
)

fig.write_html(str(CLINICAL_FIGURES_DIR / "global_model_comparison_clinical.html"))
fig.show()

print("✅ Clinical comparison figure saved")

✅ Clinical comparison figure saved


## Cadre clinique de l’analyse

Dans un contexte de triage textuel non diagnostique, toutes les erreurs ne se valent pas.

Une attention particulière est portée aux :
- **faux négatifs** sur les classes critiques,
- confusions entre classes proches,
- cas ambigus pouvant poser un risque d’interprétation.

Les classes prioritaires dans cette analyse sont :
- `Bipolar`
- `schizophrenia`

In [34]:
# ============================================================
# PHASE 4 — BUILD CLINICAL ANALYSIS TABLE
# ============================================================

clinical_analysis_df = final_predictions_df.copy()

# --- Robust column normalization ---
if "raw_text" in clinical_analysis_df.columns:
    clinical_analysis_df = clinical_analysis_df.rename(columns={"raw_text": TEXT_COL})
elif "text" in clinical_analysis_df.columns:
    clinical_analysis_df = clinical_analysis_df.rename(columns={"text": TEXT_COL})
else:
    raise ValueError("No usable text column found in final_predictions_df")

if "y_true_label" in clinical_analysis_df.columns:
    clinical_analysis_df = clinical_analysis_df.rename(columns={"y_true_label": "y_true"})
elif "y_true" not in clinical_analysis_df.columns:
    raise ValueError("No usable true label column found in final_predictions_df")

if "y_pred_label" in clinical_analysis_df.columns:
    clinical_analysis_df = clinical_analysis_df.rename(columns={"y_pred_label": "y_pred"})
elif "y_pred" not in clinical_analysis_df.columns:
    raise ValueError("No usable predicted label column found in final_predictions_df")

clinical_analysis_df["is_correct"] = (
    clinical_analysis_df["y_true"] == clinical_analysis_df["y_pred"]
)

clinical_analysis_df["error_type"] = np.where(
    clinical_analysis_df["is_correct"],
    "correct",
    "error",
)

clinical_analysis_df["is_critical_true"] = clinical_analysis_df["y_true"].isin(CRITICAL_LABELS)
clinical_analysis_df["is_critical_pred"] = clinical_analysis_df["y_pred"].isin(CRITICAL_LABELS)

clinical_analysis_df["critical_error_type"] = np.select(
    [
        (clinical_analysis_df["y_true"].isin(CRITICAL_LABELS)) &
        (clinical_analysis_df["y_pred"] == clinical_analysis_df["y_true"]),

        (clinical_analysis_df["y_true"].isin(CRITICAL_LABELS)) &
        (clinical_analysis_df["y_pred"] != clinical_analysis_df["y_true"]),

        (~clinical_analysis_df["y_true"].isin(CRITICAL_LABELS)) &
        (clinical_analysis_df["y_pred"].isin(CRITICAL_LABELS)),
    ],
    [
        "critical_correct",
        "critical_false_negative",
        "critical_false_positive",
    ],
    default="non_critical_or_other",
)

display(clinical_analysis_df.head(5))
print("✅ Clinical analysis table built")
print("Shape:", clinical_analysis_df.shape)

,body,text_used_for_model,y_true,y_pred,is_correct,error_type,is_critical_true,is_critical_pred,critical_error_type
0,My first day of JR year starts tomorrow and I cant do it Well I can but its very difficult for me mentally I was online from 7th9th grade went in person for 10th grade in a different area and have no friends I dont even have any from middle school or elementary either since I lived far away from...,My first day of JR year starts tomorrow and I cant do it Well I can but its very difficult for me mentally I was online from 7th9th grade went in person for 10th grade in a different area and have no friends I dont even have any from middle school or elementary either since I lived far away from...,Anxiety,Anxiety,True,correct,False,False,non_critical_or_other
1,Its one of those days where I feel like garbage and completely hate myself for it\n\n\n\nI dont even know where to begin\n\n\n\nI came home from work after trying to convince myself that everyone likes me or at the very least doesnt hate me But all I get instead is just a non stop paranoid strea...,Its one of those days where I feel like garbage and completely hate myself for it\n\n\n\nI dont even know where to begin\n\n\n\nI came home from work after trying to convince myself that everyone likes me or at the very least doesnt hate me But all I get instead is just a non stop paranoid strea...,Anxiety,Anxiety,True,correct,False,False,non_critical_or_other
2,a while back i stumbled across a ted talk about adhd and the speaker open his lecture by listing all the hobbies he has he had a lot but it made me realize that i have a lot too\n\nmy hobbies\n\nmusic love to listen to music also play guitar and use to play drums\n\nvideo games play dota 2 mostl...,a while back i stumbled across a ted talk about adhd and the speaker open his lecture by listing all the hobbies he has he had a lot but it made me realize that i have a lot too\n\nmy hobbies\n\nmusic love to listen to music also play guitar and use to play drums\n\nvideo games play dota 2 mostl...,ADHD,ADHD,True,correct,False,False,non_critical_or_other
3,I was having cardiac anxiety amp after 1 year of struggle more than 20 Lipid profile tests consultation with 15 cardiologists i finally gave up on the thoughts of getting heart problems amp it successfully resolved my cardiophobia hypochondria\n\nNow after 2 months something happened at my famil...,I was having cardiac anxiety amp after 1 year of struggle more than 20 Lipid profile tests consultation with 15 cardiologists i finally gave up on the thoughts of getting heart problems amp it successfully resolved my cardiophobia hypochondria\n\nNow after 2 months something happened at my famil...,Anxiety,Anxiety,True,correct,False,False,non_critical_or_other
4,My anxiety has tormented me for the most of my living time on this earth for as long as my memory allows as soon as preteen But recently its been at its worst I feel like a failure in everything that I do When my anxiety triggers its like I lose all common sense and how to act Like even driving ...,My anxiety has tormented me for the most of my living time on this earth for as long as my memory allows as soon as preteen But recently its been at its worst I feel like a failure in everything that I do When my anxiety triggers its like I lose all common sense and how to act Like even driving ...,Anxiety,Anxiety,True,correct,False,False,non_critical_or_other


✅ Clinical analysis table built
Shape: (2255, 9)


In [17]:
clinical_analysis_df.to_csv(
    CLINICAL_TABLES_DIR / "clinical_analysis_table.csv",
    index=False,
)

print("✅ Clinical analysis table saved")

✅ Clinical analysis table saved


## Analyse des erreurs

Cette section examine les erreurs du modèle champion avec une priorité donnée aux classes cliniquement sensibles.

L’objectif n’est pas seulement de mesurer la performance, mais aussi de comprendre :
- quelles erreurs sont les plus risquées,
- quelles classes sont les plus souvent confondues,
- quels exemples méritent une revue qualitative.

In [18]:
# ============================================================
# PHASE 5 — GLOBAL ERROR SUMMARY
# ============================================================

error_summary_df = pd.DataFrame({
    "metric": [
        "total_predictions",
        "correct_predictions",
        "errors",
        "accuracy_proxy",
        "critical_true_cases",
        "critical_false_negatives",
        "critical_false_positives",
    ],
    "value": [
        len(clinical_analysis_df),
        int((clinical_analysis_df["error_type"] == "correct").sum()),
        int((clinical_analysis_df["error_type"] == "error").sum()),
        round((clinical_analysis_df["is_correct"].mean()), 6),
        int(clinical_analysis_df["is_critical_true"].sum()),
        int((clinical_analysis_df["critical_error_type"] == "critical_false_negative").sum()),
        int((clinical_analysis_df["critical_error_type"] == "critical_false_positive").sum()),
    ],
})

display(error_summary_df)

error_summary_df.to_csv(
    CLINICAL_TABLES_DIR / "global_error_summary.csv",
    index=False,
)

print("✅ Global error summary saved")

,metric,value
0,total_predictions,2255.0000
1,correct_predictions,1781.0000
2,errors,474.0000
3,accuracy_proxy,0.7898
4,critical_true_cases,365.0000
5,critical_false_negatives,95.0000
6,critical_false_positives,63.0000


✅ Global error summary saved


## Faux négatifs et faux positifs critiques

Dans un usage de triage non diagnostique, les erreurs les plus sensibles sont :

- les **faux négatifs critiques** : un cas critique non reconnu comme tel,
- les **faux positifs critiques** : un cas non critique prédit comme critique.

Ces deux catégories sont isolées ci-dessous.

In [19]:
# ============================================================
# PHASE 6 — CRITICAL FALSE NEGATIVES
# ============================================================

critical_false_negatives_df = clinical_analysis_df[
    (clinical_analysis_df["y_true"].isin(CRITICAL_LABELS)) &
    (clinical_analysis_df["y_pred"] != clinical_analysis_df["y_true"])
].copy()

critical_false_negatives_df = critical_false_negatives_df.sort_values(
    by=["y_true", "y_pred"]
).reset_index(drop=True)

display(critical_false_negatives_df.head(20))

critical_false_negatives_df.to_csv(
    CLINICAL_TABLES_DIR / "critical_false_negatives.csv",
    index=False,
)

print("✅ Critical false negatives saved")
print("Count:", len(critical_false_negatives_df))

,body,text_used_for_model,y_true,y_pred,is_correct,error_type,is_critical_true,is_critical_pred,critical_error_type
0,I was thinking about this the other day everyone has a different perspective on work considering what works for some may not work for others Personally I think work is one of the only things thats kept me from doing a lot of stupid stuff Its also helped keep me out of sincerely dark places Howev...,I was thinking about this the other day everyone has a different perspective on work considering what works for some may not work for others Personally I think work is one of the only things thats kept me from doing a lot of stupid stuff Its also helped keep me out of sincerely dark places Howev...,Bipolar,ADHD,False,error,True,False,critical_false_negative
1,This is kinda a last ditch effort but I was wondering you all had any ideas about how I could get samples of this medication I was getting them through my psychiatrist for awhile but then their office made a policy to stop accepting samples I called my primary care and they have the same policy ...,This is kinda a last ditch effort but I was wondering you all had any ideas about how I could get samples of this medication I was getting them through my psychiatrist for awhile but then their office made a policy to stop accepting samples I called my primary care and they have the same policy ...,Bipolar,ADHD,False,error,True,False,critical_false_negative
2,Ive always tried to explain what it feels like being bipolar and to express how extreme the emotions are of the Highs and lows Unfortunately the emotions are something that alot of people can even imagine so its hard for them to understand\n\nSo I finally came up with an answer that can explain ...,Ive always tried to explain what it feels like being bipolar and to express how extreme the emotions are of the Highs and lows Unfortunately the emotions are something that alot of people can even imagine so its hard for them to understand\n\nSo I finally came up with an answer that can explain ...,Bipolar,ADHD,False,error,True,False,critical_false_negative
3,For years I relied on mania and hypomania to be productive I learned that depressions always end and I would even medicate depression and then get off my meds when I sensed a high coming or when I knew it was possible Now I have stopped doing that which is for the best bc mania isnt as great as ...,For years I relied on mania and hypomania to be productive I learned that depressions always end and I would even medicate depression and then get off my meds when I sensed a high coming or when I knew it was possible Now I have stopped doing that which is for the best bc mania isnt as great as ...,Bipolar,ADHD,False,error,True,False,critical_false_negative
4,will start by saying that i am not officially diagnosed yet but multiple psychiatrists said i most likely have it and i have all the symptoms\n\nbit of background first I am 23 years old was born with a very rare genetic condition called HSAN it means i dont feel pain This has resulted in me bei...,will start by saying that i am not officially diagnosed yet but multiple psychiatrists said i most likely have it and i have all the symptoms\n\nbit of background first I am 23 years old was born with a very rare genetic condition called HSAN it means i dont feel pain This has resulted in me bei...,Bipolar,ADHD,False,error,True,False,critical_false_negative
5,So tldr I was diagnosed with bipolar and ADHD and am currently taking zoloft 1year and Abilify 1month Ive been waking up at 23 am and not able to go back to sleep I started taking melatonin when I wake up at 2 am to help me sleep more but it just makes me even more sleepy without being able to s...,So tldr I was diagnosed with bipolar and ADHD and am currently taking zoloft 1year and Abilify 1month Ive been waking up at 23 am and not able to go back to sleep I started taking melatonin when I wake up at 2 am to help me sleep more but it just makes me even more sleepy

✅ Critical false negatives saved
Count: 95


In [20]:
# ============================================================
# PHASE 7 — CRITICAL FALSE POSITIVES
# ============================================================

critical_false_positives_df = clinical_analysis_df[
    (~clinical_analysis_df["y_true"].isin(CRITICAL_LABELS)) &
    (clinical_analysis_df["y_pred"].isin(CRITICAL_LABELS))
].copy()

critical_false_positives_df = critical_false_positives_df.sort_values(
    by=["y_pred", "y_true"]
).reset_index(drop=True)

display(critical_false_positives_df.head(20))

critical_false_positives_df.to_csv(
    CLINICAL_TABLES_DIR / "critical_false_positives.csv",
    index=False,
)

print("✅ Critical false positives saved")
print("Count:", len(critical_false_positives_df))

,body,text_used_for_model,y_true,y_pred,is_correct,error_type,is_critical_true,is_critical_pred,critical_error_type
0,a friend ive not seen since we were 5 or 15 i dunno time is weird sent me a message on facebook back in december 2019 i dont use fb anymore so i didnt see it til march of this year\n\nat first my rsd was like its been too long shes not interested anymore its no use shell not text back but i thou...,a friend ive not seen since we were 5 or 15 i dunno time is weird sent me a message on facebook back in december 2019 i dont use fb anymore so i didnt see it til march of this year\n\nat first my rsd was like its been too long shes not interested anymore its no use shell not text back but i thou...,ADHD,Bipolar,False,error,False,True,critical_false_positive
1,he blames it all on add he just spends his time socializing with his friends on his phone and doing social club activities he prioritizes his club activities and thinks they will be better for him than grades now he says he doesnt need to take the sat again because it doesnt matter i got no idea...,he blames it all on add he just spends his time socializing with his friends on his phone and doing social club activities he prioritizes his club activities and thinks they will be better for him than grades now he says he doesnt need to take the sat again because it doesnt matter i got no idea...,ADHD,Bipolar,False,error,False,True,critical_false_positive
2,does anyone lose interest in their relationship after a while theres nothing wrong with my so and hes definitely trying his best but im simply having a hard time staying interested this has happened in all of my relationships no matter how long or short they are and im worried i wont be able to ...,does anyone lose interest in their relationship after a while theres nothing wrong with my so and hes definitely trying his best but im simply having a hard time staying interested this has happened in all of my relationships no matter how long or short they are and im worried i wont be able to ...,ADHD,Bipolar,False,error,False,True,critical_false_positive
3,Its crazy how our perspective is so different towards different people Well mine I guess,Its crazy how our perspective is so different towards different people Well mine I guess,Anxiety,Bipolar,False,error,False,True,critical_false_positive
4,Ok I know how it sounds but I am so worried about my ex right now I cant calm down So my ex broke up with me at the end of May and I was heartbroken but soon got over it We have been dating for around a year and a half and the whole time I was always worried he was depressed or sad he wasnt tell...,Ok I know how it sounds but I am so worried about my ex right now I cant calm down So my ex broke up with me at the end of May and I was heartbroken but soon got over it We have been dating for around a year and a half and the whole time I was always worried he was depressed or sad he wasnt tell...,Anxiety,Bipolar,False,error,False,True,critical_false_positive
5,My bestie has been taking Lexapro and klonopin daily now for over 7 yrs As hard as I try to encourage him he wont look for a job or do anything to improve his situation lives at home at age 40 He lies and tells me hes trying to get a resume together but thats been 8 months or more always an excu...,My bestie has been taking Lexapro and klonopin daily now for over 7 yrs As hard as I try to encourage him he wont look for a job or do anything to improve his situation lives at home at age 40 He lies and tells me hes trying to get a resume together but thats been 8 months or more always an excu...,Anxiety,Bipolar,False,error,False,True,critical_false_positive
6,Can you take them together,Can you take them together,Anxiety,Bipolar,False,error,False,True,critical_false_positive
7,Just wanted to hear experiences with the medication lithium I know it is commonly used for bipolar or depression so I wanted to hear stories from the social anxiety part as well,Just wanted to hear experi

✅ Critical false positives saved
Count: 63


## Analyse ciblée par classe critique

Les deux classes prioritaires sont analysées séparément afin de mieux comprendre :
- leur volume,
- leur taux d’erreur,
- leurs principales confusions.

In [21]:
def build_class_error_review(df_input, target_label):
    """
    Build a compact error review table for one target class.
    """
    subset = df_input[df_input["y_true"] == target_label].copy()

    summary = pd.DataFrame({
        "metric": [
            "total_cases",
            "correct_cases",
            "errors",
            "recall_proxy",
        ],
        "value": [
            len(subset),
            int((subset["y_true"] == subset["y_pred"]).sum()),
            int((subset["y_true"] != subset["y_pred"]).sum()),
            round((subset["y_true"] == subset["y_pred"]).mean(), 6) if len(subset) else 0.0,
        ],
    })

    confusion = (
        subset["y_pred"]
        .value_counts()
        .rename_axis("predicted_label")
        .reset_index(name="count")
    )

    return summary, confusion, subset

In [22]:
# ============================================================
# PHASE 8 — BIPOLAR REVIEW
# ============================================================

bipolar_summary_df, bipolar_confusion_df, bipolar_cases_df = build_class_error_review(
    clinical_analysis_df,
    "Bipolar",
)

display(bipolar_summary_df)
display(bipolar_confusion_df)

bipolar_summary_df.to_csv(
    CLINICAL_TABLES_DIR / "bipolar_summary_review.csv",
    index=False,
)
bipolar_confusion_df.to_csv(
    CLINICAL_TABLES_DIR / "bipolar_confusion_review.csv",
    index=False,
)

print("✅ Bipolar review saved")

,metric,value
0,total_cases,365.000000
1,correct_cases,270.000000
2,errors,95.000000
3,recall_proxy,0.739726


,predicted_label,count
0,Bipolar,270
1,Depression,34
2,BPD,22
3,schizophrenia,14
4,Anxiety,12
5,ADHD,7
6,Autism,6


✅ Bipolar review saved


In [23]:
# ============================================================
# PHASE 9 — SCHIZOPHRENIA REVIEW
# ============================================================

schizophrenia_summary_df, schizophrenia_confusion_df, schizophrenia_cases_df = build_class_error_review(
    clinical_analysis_df,
    "Schizophrenia",
)

display(schizophrenia_summary_df)
display(schizophrenia_confusion_df)

schizophrenia_summary_df.to_csv(
    CLINICAL_TABLES_DIR / "schizophrenia_summary_review.csv",
    index=False,
)
schizophrenia_confusion_df.to_csv(
    CLINICAL_TABLES_DIR / "schizophrenia_confusion_review.csv",
    index=False,
)

print("✅ Schizophrenia review saved")

,metric,value
0,total_cases,0.0
1,correct_cases,0.0
2,errors,0.0
3,recall_proxy,0.0


,predicted_label,count


✅ Schizophrenia review saved


In [24]:
critical_confusion_plot_df = pd.concat(
    [
        bipolar_confusion_df.assign(target_class="Bipolar"),
        schizophrenia_confusion_df.assign(target_class="Schizophrenia"),
    ],
    axis=0,
    ignore_index=True,
)

critical_confusion_plot_df.to_csv(
    CLINICAL_TABLES_DIR / "critical_class_confusions_plot_data.csv",
    index=False,
)

fig = px.bar(
    critical_confusion_plot_df,
    x="predicted_label",
    y="count",
    color="target_class",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    title="Confusions principales pour les classes critiques",
)

fig.update_layout(
    xaxis_title="Label prédit",
    yaxis_title="Nombre de cas",
    title_x=0.5,
)

fig.write_html(str(CLINICAL_FIGURES_DIR / "critical_class_confusions.html"))
fig.show()

print("✅ Critical class confusion figure saved")

✅ Critical class confusion figure saved


## Paires de confusion les plus fréquentes

Cette analyse recense les erreurs les plus fréquentes sous forme de couples :
- label réel,
- label prédit.

Elle permet d’identifier les confusions structurelles du modèle.

In [25]:
# ============================================================
# PHASE 10 — MOST FREQUENT CONFUSION PAIRS
# ============================================================

confusion_pairs_df = clinical_analysis_df[
    clinical_analysis_df["y_true"] != clinical_analysis_df["y_pred"]
].copy()

confusion_pairs_df = (
    confusion_pairs_df.groupby(["y_true", "y_pred"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(confusion_pairs_df.head(20))

confusion_pairs_df.to_csv(
    CLINICAL_TABLES_DIR / "most_frequent_confusion_pairs.csv",
    index=False,
)

print("✅ Confusion pairs saved")

,y_true,y_pred,count
0,Anxiety,Depression,35
1,Bipolar,Depression,34
2,Depression,Anxiety,23
3,Bipolar,BPD,22
4,BPD,Depression,20
5,Depression,schizophrenia,18
6,schizophrenia,Bipolar,18
7,BPD,Bipolar,17
8,schizophrenia,Depression,17
9,Depression,BPD,15


✅ Confusion pairs saved


## Revue qualitative des cas

Les tableaux suivants servent de base à une lecture qualitative manuelle.

Ils permettent d’inspecter :
- des faux négatifs critiques,
- des faux positifs critiques,
- des cas correctement classés dans les classes critiques.

In [26]:
# ============================================================
# PHASE 11 — CORRECT CRITICAL CASES
# ============================================================

correct_critical_cases_df = clinical_analysis_df[
    (clinical_analysis_df["y_true"].isin(CRITICAL_LABELS)) &
    (clinical_analysis_df["y_true"] == clinical_analysis_df["y_pred"])
].copy()

correct_critical_cases_df = correct_critical_cases_df.sort_values(
    by=["y_true"]
).reset_index(drop=True)

display(correct_critical_cases_df.head(5))

correct_critical_cases_df.to_csv(
    CLINICAL_TABLES_DIR / "correct_critical_cases.csv",
    index=False,
)

print("✅ Correct critical cases saved")
print("Count:", len(correct_critical_cases_df))

,body,text_used_for_model,y_true,y_pred,is_correct,error_type,is_critical_true,is_critical_pred,critical_error_type
0,My husband was diagnosed bipolar 2 a year ago but recently they changed it to 1 and added Seroquel to his meds\n\nI think the signs had been there longer But hes never been this bad I think his father passing less than a year ago was a big trigger\n\nHe has been convinced for at least the last t...,My husband was diagnosed bipolar 2 a year ago but recently they changed it to 1 and added Seroquel to his meds\n\nI think the signs had been there longer But hes never been this bad I think his father passing less than a year ago was a big trigger\n\nHe has been convinced for at least the last t...,Bipolar,Bipolar,True,correct,True,True,critical_correct
1,Every Friday we invite you to share with us one thing youre grateful for that has to do with your SO or BPrelated situation\n\nIt can be\n\nSomething your SO did or say\n\nAny sign of progress\n\nAny glimpse of hope\n\nWhatever you feel like sharing\n\nLets hear it\n\n\n\nSOME TIPS\n\nWe know it...,Every Friday we invite you to share with us one thing youre grateful for that has to do with your SO or BPrelated situation\n\nIt can be\n\nSomething your SO did or say\n\nAny sign of progress\n\nAny glimpse of hope\n\nWhatever you feel like sharing\n\nLets hear it\n\n\n\nSOME TIPS\n\nWe know it...,Bipolar,Bipolar,True,correct,True,True,critical_correct
2,The minute I wake up the anxiety hits me\n\nWait wheres my boyfriend Wait oh fuck right he left Well I have to do something about that I worked so hard to get him we were so happy hes the person I want to marry hes the only person Ive ever met who checks every single box and theyre some hard to ...,The minute I wake up the anxiety hits me\n\nWait wheres my boyfriend Wait oh fuck right he left Well I have to do something about that I worked so hard to get him we were so happy hes the person I want to marry hes the only person Ive ever met who checks every single box and theyre some hard to ...,Bipolar,Bipolar,True,correct,True,True,critical_correct
3,Ive known you for a while we made plans and started to build future together I wanted only you and didnt care about anything else Now I have to learn to live alone Fuck you fuck you really I wish I never met you I will NEVER contact you again I wish this was easy and painless for me as it was fo...,Ive known you for a while we made plans and started to build future together I wanted only you and didnt care about anything else Now I have to learn to live alone Fuck you fuck you really I wish I never met you I will NEVER contact you again I wish this was easy and painless for me as it was fo...,Bipolar,Bipolar,True,correct,True,True,critical_correct
4,Im sure someone out there can understand I have bipolar 2 and take mental health days with FMLA saving my job But Im struggling financially which just causes more stress which will send me down further My family and friends throw out the idea of going on disability but Im only 31 I hate the stig...,Im sure someone out there can understand I have bipolar 2 and take mental health days with FMLA saving my job But Im struggling financially which just causes more stress which will send me down further My family and friends throw out the idea of going on disability but Im only 31 I hate the stig...,Bipolar,Bipolar,True,correct,True,True,critical_correct


✅ Correct critical cases saved
Count: 270


In [27]:
# ============================================================
# PHASE 12 — COMPACT REVIEW EXPORTS
# ============================================================

critical_false_negatives_df.head(50).to_csv(
    CLINICAL_TABLES_DIR / "critical_false_negatives_top50.csv",
    index=False,
)

critical_false_positives_df.head(50).to_csv(
    CLINICAL_TABLES_DIR / "critical_false_positives_top50.csv",
    index=False,
)

correct_critical_cases_df.head(50).to_csv(
    CLINICAL_TABLES_DIR / "correct_critical_cases_top50.csv",
    index=False,
)

print("✅ Compact review exports saved")

✅ Compact review exports saved


## Synthèse intermédiaire de l’analyse d’erreurs

À ce stade, le notebook permet déjà de documenter :

- les performances globales du champion,
- la nature des erreurs critiques,
- les principales confusions structurelles,
- les cas à relire qualitativement.

La section suivante servira à formaliser une conclusion clinique et méthodologique.

## Décision finale — Modèle champion

Le modèle retenu pour l’évaluation finale est celui qui a été exporté depuis le benchmark classique.

### Justification
Ce modèle a été conservé car il offre le meilleur compromis disponible entre :
- performance macro,
- rappel global,
- rappel sur les classes critiques,
- stabilité observée pendant l’évaluation.

### Positionnement
Ce modèle doit être interprété comme un outil de **triage textuel non diagnostique** et non comme un système de décision clinique autonome.

In [28]:
# ============================================================
# PHASE 13 — DECISION MEMO TABLE
# ============================================================

if {"metric", "value"}.issubset(final_test_metrics.columns):
    final_metric_lookup = dict(zip(final_test_metrics["metric"], final_test_metrics["value"]))
else:
    first_row = final_test_metrics.iloc[0].to_dict()
    final_metric_lookup = {
        "champion_model": first_row.get("champion_model", CHAMPION_MODEL_NAME),
        "text_variant": first_row.get("text_variant", CHAMPION_TEXT_VARIANT),
        "f1_macro": first_row.get("f1_macro", "N/A"),
        "recall_macro": first_row.get("recall_macro", "N/A"),
        "critical_recall": first_row.get("critical_recall", "N/A"),
    }

decision_memo_df = pd.DataFrame({
    "item": [
        "Champion model",
        "Text variant",
        "Final test f1_macro",
        "Final test recall_macro",
        "Final test critical_recall",
        "Critical classes",
    ],
    "value": [
        final_metric_lookup.get("champion_model", CHAMPION_MODEL_NAME),
        final_metric_lookup.get("text_variant", CHAMPION_TEXT_VARIANT),
        final_metric_lookup.get("f1_macro", "N/A"),
        final_metric_lookup.get("recall_macro", "N/A"),
        final_metric_lookup.get("critical_recall", "N/A"),
        ", ".join(CRITICAL_LABELS),
    ],
})

display(decision_memo_df)

decision_memo_df.to_csv(
    CLINICAL_TABLES_DIR / "decision_memo_table.csv",
    index=False,
)

print("✅ Decision memo table saved")

,item,value
0,Champion model,LinearSVC_balanced
1,Text variant,raw
2,Final test f1_macro,0.778927
3,Final test recall_macro,0.778686
4,Final test critical_recall,0.698068
5,Critical classes,"Bipolar, Schizophrenia"


✅ Decision memo table saved


## Limites cliniques et méthodologiques

Malgré ses performances, ce pipeline présente plusieurs limites importantes :

### 1. Usage non diagnostique
Le système n’a pas vocation à établir un diagnostic psychiatrique.

### 2. Risque de faux négatifs critiques
Certains cas critiques peuvent ne pas être détectés correctement.

### 3. Dépendance au signal textuel
Le modèle ne prend pas en compte :
- l’historique clinique,
- l’évaluation médicale,
- le contexte social ou biologique,
- la temporalité réelle des symptômes.

### 4. Limites du dataset
Les labels reflètent la structure du dataset et non une validation clinique indépendante.

### 5. Généralisation hors distribution
Les performances peuvent diminuer sur des textes différents du corpus d’entraînement.

In [29]:
# ============================================================
# PHASE 14 — CLINICAL LIMITATIONS TABLE
# ============================================================

clinical_limitations_df = pd.DataFrame({
    "limitation": [
        "Non-diagnostic use",
        "Critical false negatives remain possible",
        "Text-only modeling",
        "Dataset labels are not clinical diagnoses",
        "Potential out-of-distribution degradation",
    ],
    "practical_impact": [
        "Model output must not replace clinician judgment",
        "Some high-risk cases may be missed",
        "Important non-textual context is absent",
        "Ground truth quality remains limited by dataset construction",
        "Performance may drop on new populations or platforms",
    ],
})

display(clinical_limitations_df)

clinical_limitations_df.to_csv(
    CLINICAL_TABLES_DIR / "clinical_limitations_table.csv",
    index=False,
)

print("✅ Clinical limitations table saved")

,limitation,practical_impact
0,Non-diagnostic use,Model output must not replace clinician judgment
1,Critical false negatives remain possible,Some high-risk cases may be missed
2,Text-only modeling,Important non-textual context is absent
3,Dataset labels are not clinical diagnoses,Ground truth quality remains limited by dataset construction
4,Potential out-of-distribution degradation,Performance may drop on new populations or platforms


✅ Clinical limitations table saved


## Recommandations

À partir des résultats observés, plusieurs recommandations peuvent être formulées pour un usage responsable du pipeline :

1. utiliser ce modèle uniquement comme outil d’aide au triage,
2. maintenir une revue humaine pour les cas critiques ou ambigus,
3. surveiller en priorité les faux négatifs sur `Bipolar` et `schizophrenia`,
4. conserver une documentation explicite des limites,
5. poursuivre l’analyse qualitative des cas mal classés.

In [30]:
# ============================================================
# PHASE 15 — RECOMMENDATIONS TABLE
# ============================================================

recommendations_df = pd.DataFrame({
    "recommendation": [
        "Use as triage support only",
        "Keep human review in the loop",
        "Monitor false negatives in critical classes",
        "Document limitations explicitly",
        "Continue qualitative error review",
    ],
    "priority": [
        "High",
        "High",
        "High",
        "High",
        "Medium",
    ],
})

display(recommendations_df)

recommendations_df.to_csv(
    CLINICAL_TABLES_DIR / "recommendations_table.csv",
    index=False,
)

print("✅ Recommendations table saved")

,recommendation,priority
0,Use as triage support only,High
1,Keep human review in the loop,High
2,Monitor false negatives in critical classes,High
3,Document limitations explicitly,High
4,Continue qualitative error review,Medium


✅ Recommendations table saved


In [31]:
# ============================================================
# PHASE 16 — FINAL CLINICAL SUMMARY
# ============================================================

if "final_metric_lookup" not in globals():
    if {"metric", "value"}.issubset(final_test_metrics.columns):
        final_metric_lookup = dict(zip(final_test_metrics["metric"], final_test_metrics["value"]))
    else:
        first_row = final_test_metrics.iloc[0].to_dict()
        final_metric_lookup = {
            "champion_model": first_row.get("champion_model", CHAMPION_MODEL_NAME),
            "text_variant": first_row.get("text_variant", CHAMPION_TEXT_VARIANT),
            "f1_macro": first_row.get("f1_macro", "N/A"),
            "recall_macro": first_row.get("recall_macro", "N/A"),
            "critical_recall": first_row.get("critical_recall", "N/A"),
        }

total_cases = len(clinical_analysis_df)
total_errors = int((clinical_analysis_df["y_true"] != clinical_analysis_df["y_pred"]).sum())
critical_fn_count = int((clinical_analysis_df["critical_error_type"] == "critical_false_negative").sum())
critical_fp_count = int((clinical_analysis_df["critical_error_type"] == "critical_false_positive").sum())

final_clinical_summary_df = pd.DataFrame({
    "item": [
        "Champion model",
        "Text variant",
        "Total reviewed predictions",
        "Total errors",
        "Critical false negatives",
        "Critical false positives",
        "Final test f1_macro",
        "Final test recall_macro",
        "Final test critical_recall",
    ],
    "value": [
        final_metric_lookup.get("champion_model", CHAMPION_MODEL_NAME),
        final_metric_lookup.get("text_variant", CHAMPION_TEXT_VARIANT),
        total_cases,
        total_errors,
        critical_fn_count,
        critical_fp_count,
        final_metric_lookup.get("f1_macro", "N/A"),
        final_metric_lookup.get("recall_macro", "N/A"),
        final_metric_lookup.get("critical_recall", "N/A"),
    ],
})

display(final_clinical_summary_df)

final_clinical_summary_df.to_csv(
    CLINICAL_TABLES_DIR / "final_clinical_summary.csv",
    index=False,
)

print("✅ Final clinical summary saved")

,item,value
0,Champion model,LinearSVC_balanced
1,Text variant,raw
2,Total reviewed predictions,2255
3,Total errors,474
4,Critical false negatives,95
5,Critical false positives,63
6,Final test f1_macro,0.778927
7,Final test recall_macro,0.778686
8,Final test critical_recall,0.698068


✅ Final clinical summary saved


In [32]:
summary_plot_df = final_clinical_summary_df[
    final_clinical_summary_df["item"].isin([
        "Total errors",
        "Critical false negatives",
        "Critical false positives",
    ])
].copy()

summary_plot_df["value"] = pd.to_numeric(summary_plot_df["value"], errors="coerce")

fig = px.bar(
    summary_plot_df,
    x="item",
    y="value",
    text="value",
    template=PLOTLY_TEMPLATE,
    title="Résumé des erreurs cliniquement sensibles",
)

fig.update_layout(
    xaxis_title="Indicateur",
    yaxis_title="Nombre de cas",
    title_x=0.5,
)

fig.write_html(str(CLINICAL_FIGURES_DIR / "clinical_error_summary.html"))
fig.show()

print("✅ Clinical error summary figure saved")

✅ Clinical error summary figure saved


In [33]:
# ============================================================
# PHASE 17 — FINAL EXPORT CHECK
# ============================================================

clinical_exports_df = pd.DataFrame({
    "artifact": [
        "clinical_analysis_table.csv",
        "global_error_summary.csv",
        "critical_false_negatives.csv",
        "critical_false_positives.csv",
        "most_frequent_confusion_pairs.csv",
        "decision_memo_table.csv",
        "clinical_limitations_table.csv",
        "recommendations_table.csv",
        "final_clinical_summary.csv",
    ],
    "path": [
        str(CLINICAL_TABLES_DIR / "clinical_analysis_table.csv"),
        str(CLINICAL_TABLES_DIR / "global_error_summary.csv"),
        str(CLINICAL_TABLES_DIR / "critical_false_negatives.csv"),
        str(CLINICAL_TABLES_DIR / "critical_false_positives.csv"),
        str(CLINICAL_TABLES_DIR / "most_frequent_confusion_pairs.csv"),
        str(CLINICAL_TABLES_DIR / "decision_memo_table.csv"),
        str(CLINICAL_TABLES_DIR / "clinical_limitations_table.csv"),
        str(CLINICAL_TABLES_DIR / "recommendations_table.csv"),
        str(CLINICAL_TABLES_DIR / "final_clinical_summary.csv"),
    ],
})

display(clinical_exports_df)

clinical_exports_df.to_csv(
    CLINICAL_TABLES_DIR / "clinical_exports_manifest.csv",
    index=False,
)

print("✅ Clinical exports manifest saved")

,artifact,path
0,clinical_analysis_table.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\clinical_analysis_table.csv
1,global_error_summary.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\global_error_summary.csv
2,critical_false_negatives.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\critical_false_negatives.csv
3,critical_false_positives.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\critical_false_positives.csv
4,most_frequent_confusion_pairs.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\most_frequent_confusion_pairs.csv
5,decision_memo_table.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\decision_memo_table.csv
6,clinical_limitations_table.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\clinical_limitations_table.csv
7,recommendations_table.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\recommendations_table.csv
8,final_clinical_summary.csv,C:\Users\anafi\Desktop\final_project_jedha\reports\tables\clinical\final_clinical_summary.csv


✅ Clinical exports manifest saved


## Conclusion finale

Le modèle champion présente des performances globalement solides pour un usage de triage textuel non diagnostique.

L’analyse d’erreurs montre néanmoins que :
- certaines confusions persistent,
- des faux négatifs critiques restent possibles,
- une supervision humaine demeure indispensable.

Ce notebook clôt donc l’évaluation clinique en fournissant :
- une synthèse globale,
- une revue ciblée des erreurs,
- des recommandations d’usage,
- une documentation claire des limites.

#-----------

#----------

## Avertissement clinique

Ce projet est un travail de data science appliquée et ne constitue en aucun cas un outil médical validé.

### Important

- Ce modèle ne fournit **aucun diagnostic médical**.
- Il s'agit uniquement d’un **outil de triage basé sur du texte**.
- Toute décision clinique doit être prise par un professionnel de santé qualifié.

### Risques identifiés

- Faux négatifs possibles sur des cas critiques.
- Interprétation limitée aux signaux textuels.
- Dépendance aux données d’entraînement.

### Usage recommandé

Ce système peut être utilisé comme :
- outil d’aide à la priorisation,
- support exploratoire,
- base de recherche.

Mais jamais comme :
- outil de diagnostic,
- système autonome de décision clinique.

